In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
df = pd.read_csv('../data/df_merged.csv')
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")

# ==========================================
# 2. ENHANCED DATASET OVERVIEW (CLEAN TABLE)
# ==========================================
print("\n--- 2. CLEAN DESCRIPTIVE STATISTICS ---")

# 1. Define the columns we want to look at
stats_cols = ['ind_prod_const_sa', 'exports_gdp_pct', 'usa_imp_sa']

# 2. Create the descriptive table
# .T swaps rows and columns for better fit
# .drop('count') removes the 228.00 line which isn't very useful
clean_stats = df.groupby('country')[stats_cols].describe().T.drop('count', level=1)

# 3. Apply Styling:
# - Precision(2) removes the 'e+02' scientific notation
# - Background gradient makes it look like a professional report
styled_table = clean_stats.style.format(precision=2, thousands=",") \
                          .background_gradient(cmap='Blues') \
                          .set_properties(**{'text-align': 'center', 'font-size': '12pt'})

# Display the result
display(styled_table)

# =========================
# 3. DATE + SORTING
# =========================
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
df = df.sort_values(['country', 'date'])

# =========================
# 4. FEATURE ENGINEERING
# =========================
# IP Growth (YoY)
df['ip_yoy_growth'] = df.groupby('country')['ind_prod_const_sa'].pct_change(12) * 100

# US Benchmark Growth (YoY)
us = df.groupby('date')['usa_imp_sa'].first().reset_index()
us['usa_imp_yoy_growth'] = us['usa_imp_sa'].pct_change(12) * 100

# Merge back and clean
df = df.merge(us[['date', 'usa_imp_yoy_growth']], on='date')
df = df.dropna(subset=['ip_yoy_growth', 'usa_imp_yoy_growth'])

# =========================
# 5. TRADE DEPENDENCY
# =========================
trade_dep = df.groupby('country')['exports_gdp_pct'].mean().reset_index().sort_values('exports_gdp_pct')

plt.figure(figsize=(10, 5))
sns.barplot(data=trade_dep, x='exports_gdp_pct', y='country', palette="Blues_d")
plt.title("Trade Openness: Exports as % of GDP", fontsize=14, fontweight='bold')
plt.xlabel("Average Exports/GDP (%)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# =========================
# 6. US IMPORT TREND (FIXED)
# =========================
us_ts = df.groupby('date')[['usa_imp_sa','usa_imp_yoy_growth']].first()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Upper Graph
ax1.plot(us_ts.index, us_ts['usa_imp_sa'], color='#2c3e50', linewidth=2)
ax1.set_title("US Imports Level ($ Value)", fontsize=12, fontweight='bold')
ax1.fill_between(us_ts.index, us_ts['usa_imp_sa'], color='#2c3e50', alpha=0.1)

# Lower Graph - FIXED COLORS
# We explicitly set colors so they don't default to white/invisible
colors = ['#27ae60' if x > 0 else '#e74c3c' for x in us_ts['usa_imp_yoy_growth']]

ax2.bar(us_ts.index, us_ts['usa_imp_yoy_growth'], color=colors, width=20) # width adjusted for monthly data
ax2.axhline(0, color='black', linewidth=1) # Makes the zero-line clear
ax2.set_title("US Imports YoY Growth (%)", fontsize=12, fontweight='bold')
ax2.set_ylabel("Percentage (%)")

plt.tight_layout()
plt.show()

# =========================
# 7. INDUSTRIAL PRODUCTION LEVELS
# =========================
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='date', y='ind_prod_const_sa', hue='country', palette="magma", linewidth=2)
plt.title("Industrial Production Levels by Country", fontsize=14, fontweight='bold')
plt.ylabel("Index / Level")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# =========================
# 8. IP GROWTH TIME SERIES
# =========================
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='date', y='ip_yoy_growth', hue='country', palette="viridis")
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title("Industrial Production Growth (YoY %)", fontsize=14, fontweight='bold')
plt.ylabel("% Change")
plt.show()

# ==========================================
# 9. GROWTH DISTRIBUTIONS (FORCE SEPARATION)
# ==========================================
print("\n--- 9. GROWTH DISTRIBUTION PROFILES ---")

# 1. Create the grid
g = sns.FacetGrid(df, col="country", hue="country", col_wrap=4, height=4, aspect=1.1)
g.map(sns.histplot, "ip_yoy_growth", kde=True, alpha=0.5)

# 2. Add padding to individual subplot titles to move them down away from the top
g.set_titles("{col_name}", fontweight='bold', pad=20)
g.set_axis_labels("YoY Growth %", "Count")

# 3. Adjust the entire grid down (top=0.85) and increase vertical spacing (hspace)
plt.subplots_adjust(top=0.85, hspace=0.6, wspace=0.3)

# 4. Position the main title at the very top (y=1.02)
g.fig.suptitle('Country-Specific IP Growth Distributions',
               fontsize=18,
               fontweight='bold',
               y=1.02)

plt.show()

# =========================
# 10. BOXPLOT (COMPARISON)
# =========================
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='country', y='ip_yoy_growth', palette="Set3")
sns.stripplot(data=df, x='country', y='ip_yoy_growth', color='black', size=2, alpha=0.3)
plt.title("IP Growth Variance Comparison", fontsize=14)
plt.show()

# ==================================
# 11. DYNAMIC ROLLING CORRELATION
# ==================================
print("\n11. DYNAMIC ROLLING CORRELATIONS")
plt.figure(figsize=(12, 6))
for c in df['country'].unique():
    sub = df[df['country'] == c].set_index('date').sort_index()
    rolling_corr = sub['ip_yoy_growth'].rolling(12).corr(sub['usa_imp_yoy_growth'])
    plt.plot(rolling_corr, label=f"{c}")

plt.axhline(0, color='black', linestyle='--', alpha=0.4)
plt.title("12-Month Rolling Correlation with US Demand", fontsize=14)
plt.ylabel("Pearson Correlation")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# ==================================
# 12. SCATTER + REGRESSION PLOTS
# ==================================
g = sns.lmplot(data=df, x='usa_imp_yoy_growth', y='ip_yoy_growth', col='country',
               hue='country', col_wrap=4, height=3, scatter_kws={'alpha':0.4})
g.set_titles("{col_name}")
g.fig.suptitle("Regression Analysis: US Demand vs Domestic IP", y=1.02, fontsize=16)
plt.show()

# ==================================
# 13. REGRESSION MODEL COMPARISON
# ==================================
print("\n13. REGRESSION PERFORMANCE SUMMARY")
results = {}
model_metrics = []

for c in df['country'].unique():
    sub = df[df['country']==c]
    X = sm.add_constant(sub['usa_imp_yoy_growth'])
    y = sub['ip_yoy_growth']
    model = sm.OLS(y, X).fit()
    results[c] = model

    model_metrics.append({
        'Country': c,
        'Sensitivity (Beta)': model.params['usa_imp_yoy_growth'],
        'R-Squared': model.rsquared,
        'P-Value': model.pvalues['usa_imp_yoy_growth']
    })

comparison_df = pd.DataFrame(model_metrics).sort_values('Sensitivity (Beta)', ascending=False)
display(comparison_df.style.background_gradient(cmap='RdYlGn'))

# =========================
# 14. SHOCK ANALYSIS
# =========================
df['shock'] = df['usa_imp_yoy_growth'] < -5
shock_analysis = df.groupby(['country','shock'])['ip_yoy_growth'].mean().unstack()

shock_analysis.plot(kind='bar', color=['#3498db', '#e74c3c'], figsize=(10, 5))
plt.title("Impact of US Demand Shocks (Growth < -5%)", fontweight='bold')
plt.ylabel("Mean IP Growth (%)")
plt.legend(["Normal Period", "Shock Period"])
plt.xticks(rotation=45)
plt.show()

# =========================
# 15. HEATMAP (COUNTRY RELATION)
# =========================
pivot = df.pivot_table(index='date', columns='country', values='ip_yoy_growth')
pivot['US Imports Growth'] = df.groupby('date')['usa_imp_yoy_growth'].first()
pivot.columns = [c if c == 'US Imports Growth' else f'{c} IP' for c in pivot.columns]
corr_matrix = pivot.corr()

# Use a single neutral colour so it looks like a table, not a heatmap
from matplotlib.colors import ListedColormap
neutral_cmap = ListedColormap(['#EDF2F7'])

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap=neutral_cmap, cbar=False,
            linewidths=2, linecolor='white',
            annot_kws={'fontsize': 13, 'fontweight': 'bold', 'color': '#1E293B'},
            square=True, ax=ax)

ax.set_title('Correlation Matrix\nUS Import Demand & ASEAN IP Growth (YoY %)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

"""
================================================================================
NEW CHARTS FOR ENHANCED PRESENTATION
================================================================================
Add this code AFTER your existing code in Copy_of_DA_code.ipynb
Make sure df is already loaded and ip_yoy_growth / usa_imp_yoy_growth exist
================================================================================
"""

# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 1: CORRELATION HEATMAP — Full Variable Matrix
# (For Slide: Correlation Heatmap - External Shocks & Asian Production)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 1: FULL CORRELATION HEATMAP")
print("="*70)

heatmap_cols = ['ind_prod_const_sa', 'usa_imp_sa', 'exports_gdp_pct',
                'exchnage_rate_vs_usd', 'cpi_yoy_nsa', 'gdp_per_capita_ppp']
heatmap_labels = ['ind_prod_const_sa', 'usa_imp_sa', 'exports_gdp_pct',
                  'exchange_rate_vs_usd', 'cpi_yoy_nsa', 'gdp_per_capita_ppp']

corr_full = df[heatmap_cols].corr()
corr_full.index = heatmap_labels
corr_full.columns = heatmap_labels

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_full, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=1.5, linecolor='white',
            annot_kws={'fontsize': 12, 'fontweight': 'bold'},
            vmin=-1, vmax=1, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': ''})
ax.set_title('Correlation Heatmap: External Shocks & Asian Production',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig('chart_correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Saved: chart_correlation_heatmap.png")


# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 2: HIGH vs LOW TRADE DEPENDENCY — Split Regression
# (For Slide: High vs Low Trade Dependency: Split Regression)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 2: HIGH vs LOW DEPENDENCY SPLIT REGRESSION")
print("="*70)

# Classify countries
high_dep = ['Singapore', 'Malaysia']
low_dep  = ['Philippines', 'Thailand']

df['dependency_group'] = df['country'].apply(
    lambda x: 'High Dependency' if x in high_dep else 'Low Dependency'
)

# Convert IP to billions if not already done
if 'ind_prod_const_sa_billions' not in df.columns:
    df['ind_prod_const_sa_billions'] = df['ind_prod_const_sa'] / 1e9
if 'usa_imp_sa_thousands' not in df.columns:
    df['usa_imp_sa_thousands'] = df['usa_imp_sa'] / 1e3

# --- GROUP 1: HIGH DEPENDENCY ---
print("\n" + "="*60)
print("  GROUP 1: HIGH DEPENDENCY COUNTRIES (Singapore, Malaysia)")
print("="*60)

df_high = df[df['country'].isin(high_dep)].copy()
X_high = sm.add_constant(df_high[['usa_imp_sa_thousands', 'cpi_yoy_nsa', 'exchnage_rate_vs_usd']])
y_high = df_high['ind_prod_const_sa_billions']
model_high = sm.OLS(y_high, X_high).fit()

print(f"\n{'─'*60}")
print(f"  Coefficients:")
print(f"{'─'*60}")
for var in model_high.params.index:
    print(f"  {var:<30} coef={model_high.params[var]:.4f}  p={model_high.pvalues[var]:.4f}")
print(f"\n  Overall Model Fit (R-squared): {model_high.rsquared:.3f}")
print(f"  Overall Model Strength (F-statistic): {model_high.fvalue:.1f}")

# --- GROUP 2: LOW DEPENDENCY ---
print("\n" + "="*60)
print("  GROUP 2: LOW DEPENDENCY COUNTRIES (Philippines, Thailand)")
print("="*60)

df_low = df[df['country'].isin(low_dep)].copy()
X_low = sm.add_constant(df_low[['usa_imp_sa_thousands', 'cpi_yoy_nsa', 'exchnage_rate_vs_usd']])
y_low = df_low['ind_prod_const_sa_billions']
model_low = sm.OLS(y_low, X_low).fit()

print(f"\n{'─'*60}")
print(f"  Coefficients:")
print(f"{'─'*60}")
for var in model_low.params.index:
    print(f"  {var:<30} coef={model_low.params[var]:.4f}  p={model_low.pvalues[var]:.4f}")
print(f"\n  Overall Model Fit (R-squared): {model_low.rsquared:.3f}")
print(f"  Overall Model Strength (F-statistic): {model_low.fvalue:.1f}")

# --- COMPARISON ---
beta_high = model_high.params['usa_imp_sa_thousands']
beta_low  = model_low.params['usa_imp_sa_thousands']
sensitivity_diff = ((beta_high - beta_low) / beta_low) * 100

print("\n" + "="*60)
print("  COMPARISON SUMMARY")
print("="*60)
print(f"  High Dependency β (usa_imp): {beta_high:.4f}")
print(f"  Low Dependency  β (usa_imp): {beta_low:.4f}")
print(f"  Difference: High dependency is {sensitivity_diff:.0f}% more sensitive")
print(f"  High R²: {model_high.rsquared:.3f}  vs  Low R²: {model_low.rsquared:.3f}")


# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 3: MACROECONOMIC VOLATILITY — High vs Low Box Plot
# (For Slide: H2 Results — Trade Dependency Amplifies Shocks)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 3: VOLATILITY COMPARISON BOX PLOT")
print("="*70)

# Calculate 12-month rolling volatility
df_sorted = df.sort_values(['country', 'date']).copy()
df_sorted['rolling_vol'] = df_sorted.groupby('country')['ip_yoy_growth'].transform(
    lambda x: x.rolling(12).std()
)

fig, ax = plt.subplots(figsize=(8, 5))
colors_box = {'High Dependency': '#F59E0B', 'Low Dependency': '#3B82F6'}

sns.boxplot(data=df_sorted.dropna(subset=['rolling_vol']),
            x='dependency_group', y='rolling_vol',
            palette=colors_box, width=0.5, ax=ax)

ax.set_title('Macroeconomic Volatility: High vs. Low Trade Dependency',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Structural Economic Profile', fontsize=11)
ax.set_ylabel('12-Month Rolling Volatility (Std Dev of MoM Growth)', fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('chart_volatility_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Saved: chart_volatility_boxplot.png")


# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 4: TRADE DEPENDENCY vs ECONOMIC VOLATILITY (Scatter + Trend)
# (For Slide: Structural Risk — Trade Dependency vs Volatility)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 4: STRUCTURAL RISK SCATTER")
print("="*70)

# Calculate annual volatility per country per year
annual_vol = df.groupby(['country', 'year']).agg(
    vol=('ip_yoy_growth', 'std'),
    trade_dep=('exports_gdp_pct', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(annual_vol['trade_dep'], annual_vol['vol'],
                     c=annual_vol['trade_dep'], cmap='Blues', alpha=0.7,
                     edgecolors='#1B2B3F', linewidth=0.5, s=50)

# Trend line
z = np.polyfit(annual_vol['trade_dep'], annual_vol['vol'], 1)
p = np.poly1d(z)
x_line = np.linspace(annual_vol['trade_dep'].min(), annual_vol['trade_dep'].max(), 100)
ax.plot(x_line, p(x_line), color='#EF4444', linewidth=2.5, alpha=0.8)
ax.fill_between(x_line, p(x_line) - 1, p(x_line) + 1, color='#EF4444', alpha=0.1)

ax.set_title('Structural Risk: Trade Dependency vs. Economic Volatility',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Trade Dependency (Exports as % of GDP)', fontsize=11)
ax.set_ylabel('Macroeconomic Volatility (12M Rolling Std Dev)', fontsize=11)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('chart_structural_risk.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Saved: chart_structural_risk.png")


# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 5: AMPLIFICATION EFFECT — Indexed Production by Dependency Group
# (For Slide: H2 Results — Trade Dependency Amplifies Shocks)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 5: AMPLIFICATION EFFECT SCATTER")
print("="*70)

# Index production to base = 100 for each country
df_idx = df.copy()
base_vals = df_idx.groupby('country')['ind_prod_const_sa'].transform('first')
df_idx['ip_indexed'] = (df_idx['ind_prod_const_sa'] / base_vals) * 100

fig, ax = plt.subplots(figsize=(10, 6))

for grp, color, label in [('High Dependency', '#3B82F6', 'High Trade Dependency'),
                            ('Low Dependency', '#F59E0B', 'Low Trade Dependency')]:
    sub = df_idx[df_idx['dependency_group'] == grp]
    ax.scatter(sub['usa_imp_sa'], sub['ip_indexed'], alpha=0.3, s=15, color=color, label=label)
    # Trend line
    z = np.polyfit(sub['usa_imp_sa'], sub['ip_indexed'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sub['usa_imp_sa'].min(), sub['usa_imp_sa'].max(), 100)
    ax.plot(x_line, p(x_line), color=color, linewidth=2.5)

ax.set_title('The Amplification Effect: How Trade Dependency Changes the Impact of US Shocks\n'
             '(Industrial Production Indexed to Base=100)', fontsize=13, fontweight='bold')
ax.set_xlabel('US Imports (External Demand Shock)', fontsize=11)
ax.set_ylabel('Industrial Production (Index, Start = 100)', fontsize=11)
ax.legend(title='Economic Profile', fontsize=10)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('chart_amplification_effect.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Saved: chart_amplification_effect.png")


# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 6: TRADE DEPENDENCY vs BETA SENSITIVITY (Cross-Country r=0.978)
# (For Slide: Does Trade Dependency Amplify US Demand Shocks?)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 6: TRADE DEPENDENCY vs BETA SENSITIVITY")
print("="*70)

# Get beta per country (from existing results dict if available)
beta_data = []
for c in df['country'].unique():
    sub = df[df['country'] == c]
    X = sm.add_constant(sub['usa_imp_yoy_growth'])
    y = sub['ip_yoy_growth']
    model = sm.OLS(y, X).fit()
    avg_trade = sub['exports_gdp_pct'].mean()
    beta_data.append({
        'Country': c,
        'Avg_Trade_Dep': avg_trade,
        'Beta': model.params['usa_imp_yoy_growth']
    })

beta_df = pd.DataFrame(beta_data)
r_val = beta_df['Avg_Trade_Dep'].corr(beta_df['Beta'])

fig, ax = plt.subplots(figsize=(8, 6))
colors_country = {'Malaysia': '#1B2B3F', 'Singapore': '#0891B2',
                   'Philippines': '#10B981', 'Thailand': '#EF4444'}

for _, row in beta_df.iterrows():
    ax.scatter(row['Avg_Trade_Dep'], row['Beta'], s=150, zorder=5,
               color=colors_country.get(row['Country'], '#333'),
               edgecolors='white', linewidth=2)
    ax.annotate(row['Country'], (row['Avg_Trade_Dep'], row['Beta']),
                fontsize=11, fontweight='bold',
                xytext=(10, 5), textcoords='offset points')

# Trend line
z = np.polyfit(beta_df['Avg_Trade_Dep'], beta_df['Beta'], 1)
p = np.poly1d(z)
x_line = np.linspace(beta_df['Avg_Trade_Dep'].min() - 10, beta_df['Avg_Trade_Dep'].max() + 10, 100)
ax.plot(x_line, p(x_line), '--', color='#EF4444', linewidth=2, alpha=0.7)

ax.set_title(f'Does Trade Dependency Amplify US Demand Shocks?\n(RQ2: Cross-Country Evidence)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Average Trade Dependency (Exports as % of GDP)', fontsize=11)
ax.set_ylabel('Sensitivity to US Demand (β coefficient)', fontsize=11)

# Add r value annotation
ax.text(0.95, 0.05, f'Cross-country signal\nr = {r_val:.3f}',
        transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FEF3C7', edgecolor='#F59E0B'))

ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('chart_trade_dep_vs_beta.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ Saved: chart_trade_dep_vs_beta.png (r = {r_val:.3f})")


# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 7: HYPOTHESIS TEST — Does High Trade Dependency Reduce Stability?
# (For Slide: Volatility Regression)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 7: HYPOTHESIS TEST — TRADE DEPENDENCY vs VOLATILITY")
print("="*70)

# Regression: volatility ~ exports_gdp_pct + cpi + exchange_rate
vol_df = df_sorted.dropna(subset=['rolling_vol']).copy()

X_vol = sm.add_constant(vol_df[['exports_gdp_pct', 'cpi_yoy_nsa', 'exchnage_rate_vs_usd']])
y_vol = vol_df['rolling_vol']
model_vol = sm.OLS(y_vol, X_vol).fit()

print("\n  HYPOTHESIS TEST: Does High Trade Dependency Reduce Economic Shocks?")
print("="*60)
print(model_vol.summary())

coef_trade = model_vol.params['exports_gdp_pct']
pval_trade = model_vol.pvalues['exports_gdp_pct']

print(f"\n  Coefficient (Trade Dependency) : {coef_trade:.6f}")
print(f"  P-Value                        : {pval_trade:.4e}")
print(f"  >> Result    : {'REJECT the Null Hypothesis (H0).' if pval_trade < 0.05 else 'FAIL TO REJECT H0.'}")


# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 8: COUNTRY-LEVEL REGRESSIONS (with exchange rate — simplified)
# (For Slide: Country-Level Regression Results)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 8: COUNTRY-LEVEL REGRESSIONS (IP ~ US_Imp + Exchange Rate)")
print("="*70)

# Scale US imports for nicer coefficients
df['usa_imp_scaled'] = df['usa_imp_sa'] / 1e3

country_results_new = {}
for country in sorted(df['country'].unique()):
    sub = df[df['country'] == country].copy()
    X_c = sm.add_constant(sub[['usa_imp_scaled', 'exchnage_rate_vs_usd']])
    y_c = sub['ind_prod_const_sa_billions']
    model_c = sm.OLS(y_c, X_c).fit()
    country_results_new[country] = model_c

    print(f"\n{'='*50}")
    print(f"  Regression Results for {country}")
    print(f"{'='*50}")
    print(model_c.summary())


# ══════════════════════════════════════════════════════════════════════════════
# NEW CHART 9: BINNED SCATTER PLOTS — US Imports vs IP per Country
# (For Slide: Country-Level US Imports vs Industrial Production)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  NEW CHART 9: BINNED SCATTER PLOTS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
countries_list = ['Singapore', 'Malaysia', 'Thailand', 'Philippines']
colors_binned = ['#0891B2', '#1B2B3F', '#EF4444', '#10B981']

for ax, country, color in zip(axes.flatten(), countries_list, colors_binned):
    sub = df[df['country'] == country].copy()

    # Scatter raw data points
    ax.scatter(sub['usa_imp_sa'], sub['ind_prod_const_sa'],
               alpha=0.15, s=10, color='gray', label='Monthly Data Points')

    # Binned trend
    sub['usa_bin'] = pd.cut(sub['usa_imp_sa'], bins=20)
    binned = sub.groupby('usa_bin').agg(
        usa_mid=('usa_imp_sa', 'mean'),
        ip_mean=('ind_prod_const_sa', 'mean')
    ).dropna()

    ax.plot(binned['usa_mid'], binned['ip_mean'], 'o--',
            color=color, linewidth=2, markersize=5, label=f'{country} Binned Trend')

    ax.set_title(f'Binned Scatter Plot: {country}', fontsize=12, fontweight='bold')
    ax.set_xlabel('US Imports (USD Millions)', fontsize=9)
    ax.set_ylabel('Industrial Production', fontsize=9)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(linestyle='--', alpha=0.3)

plt.suptitle('Country-Level: US Imports vs Industrial Production', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('chart_binned_scatter.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Saved: chart_binned_scatter.png")


# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("  ✅ ALL NEW CHARTS COMPLETE")
print("  Charts saved:")
print("    1. chart_correlation_heatmap.png")
print("    2. chart_volatility_boxplot.png")
print("    3. chart_structural_risk.png")
print("    4. chart_amplification_effect.png")
print("    5. chart_trade_dep_vs_beta.png")
print("    6. chart_binned_scatter.png")
print("="*70)

# ═══════════════════════════════════════════════════════════════
# Correlation Analysis: Detailed Statistical Table
# ═══════════════════════════════════════════════════════════════
from scipy.stats import pearsonr

print("=" * 75)
print("  CORRELATION ANALYSIS: US Import Demand Growth vs IP Growth")
print("=" * 75)
print(f"\n{'Country':<15} {'r':>8} {'p-value':>14} {'N':>6} {'r²':>8} {'Sig':>6} {'Strength':>15}")
print("─" * 75)

corr_results = []
for country in sorted(df['country'].unique()):
    sub = df[df['country'] == country].dropna(subset=['ip_yoy_growth', 'usa_imp_yoy_growth'])
    r, p = pearsonr(sub['ip_yoy_growth'], sub['usa_imp_yoy_growth'])
    n = len(sub)
    r2 = r ** 2

    stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    strength = 'Strong' if abs(r) >= 0.6 else 'Moderate' if abs(r) >= 0.4 else 'Weak'

    # Human-readable p-value
    if p < 0.001:
        p_display = '< 0.001'
    elif p < 0.01:
        p_display = '< 0.01'
    elif p < 0.05:
        p_display = f'{p:.3f}'
    else:
        p_display = f'{p:.3f}'

    print(f"{country:<15} {r:>8.3f} {p_display:>14} {n:>6} {r2:>8.3f} {stars:>6} {strength:>15}")

    corr_results.append({
        'Country': country,
        'Pearson r': round(r, 3),
        'p-value': p_display,
        'N': n,
        'r² (variance explained)': round(r2, 3),
        'Significance': stars,
        'Strength': strength
    })

print("─" * 75)
print("*** p<0.001  ** p<0.01  * p<0.05  ns = not significant")

corr_table = pd.DataFrame(corr_results).set_index('Country')
display(
    corr_table.style
    .background_gradient(subset=['Pearson r'], cmap='Blues')
    .background_gradient(subset=['r² (variance explained)'], cmap='Greens')
)

# ==================================
# 16. FINAL KEY INSIGHTS
# ==================================
print("\n" + "="*50)
print("STRATEGIC INSIGHTS SUMMARY")
print("="*50)

for _, row in comparison_df.iterrows():
    c = row['Country']
    beta = row['Sensitivity (Beta)']
    r2 = row['R-Squared']

    dep_level = "High" if beta > 0.5 else "Moderate" if beta > 0.2 else "Low"
    print(f"- {c}: {dep_level} sensitivity to US demand (Beta: {beta:.2f}, Explanatory Power: {r2:.1%})")

print("\nAnalysis Complete.")